# Packages installations

### Install all these packages  : 
* df-squeezer
* scikit-learn
* lightgbm
* xgboost
* catboost[gpu]
* numpy
* matplotlib
* pandas
* pyarrow
* fastparquet
* torch
* ipykernel 
* tqdm

In [ ]:
pip install -r /kaggle/input/datassets/requirements.txt  # assurer vous de changer le chemin d'acces du requirements 

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 55.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 50.2 MB/s

### 0. Imports and requirements

In [2]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sn
# Input data files are available in the read-only "../input/" director.ai/competitions/aivkchallengey
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

from df_squeezer import df_squeezer
from sklearn.metrics import roc_auc_score, accuracy_score

%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import pandas as pd
import numpy as np
import tqdm
import seaborn as sns


import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression


pd.set_option('display.max_columns', None)
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

sys.path.append('../')

# 1. Reading Data and Memory Reduction

In [3]:

train = pd.read_parquet("/kaggle/input/datassets/train.parquet") # veillez adpter le chemin d'acces au votre 
test = pd.read_parquet("/kaggle/input/datassets/test.parquet") # veillez adpter le chemin d'acces au votre
train=df_squeezer(train, edit=True, report=False)
test=df_squeezer(test, edit=True, report=False)

# 2.  Data Preporcessing 

In [4]:
train.head()

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag,flag
0,1678548,3,8,7,17,16,9,1,9,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,1,3,2,1,0,0,0
1,2834188,11,3,2,0,7,14,8,2,5,1,0,2,13,6,16,5,4,8,1,0,1,1,1,1,2,17,0,1,1,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,1,2,4,1,0,0,0
2,811902,11,9,6,11,13,14,8,2,5,1,0,2,11,6,16,5,4,8,1,1,1,1,1,11,2,17,0,1,1,0,0,0,0,0,0,0,0,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0,0
3,836450,1,16,0,13,0,4,9,5,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0,0
4,1769024,15,9,9,4,8,1,11,1,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,1,1,0


In [5]:
train.shape

(17659833, 62)

In [14]:
train.isna().sum().sort_values(ascending=False)
#ON constate qu'il n'y pas de valeurs manquantes

id                       0
rn                       0
pre_since_opened         0
pre_since_confirmed      0
pre_pterm                0
                        ..
enc_loans_credit_type    0
enc_loans_account_cur    0
pclose_flag              0
fclose_flag              0
flag                     0
Length: 62, dtype: int64

### 2.1 Columns Analysis

In [15]:
train.columns

Index(['id', 'rn', 'pre_since_opened', 'pre_since_confirmed', 'pre_pterm',
       'pre_fterm', 'pre_till_pclose', 'pre_till_fclose',
       'pre_loans_credit_limit', 'pre_loans_next_pay_summ',
       'pre_loans_outstanding', 'pre_loans_total_overdue',
       'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_loans5',
       'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90',
       'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060',
       'is_zero_loans6090', 'is_zero_loans90', 'pre_util', 'pre_over2limit',
       'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit',
       'is_zero_maxover2limit', 'enc_paym_0', 'enc_paym_1', 'enc_paym_2',
       'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6', 'enc_paym_7',
       'enc_paym_8', 'enc_paym_9', 'enc_paym_10', 'enc_paym_11', 'enc_paym_12',
       'enc_paym_13', 'enc_paym_14', 'enc_paym_15', 'enc_paym_16',
       'enc_paym_17', 'enc_paym_18', 'enc_paym_19', 'enc_paym_20',
       'enc_paym_21', 

In [16]:
train['rn'].value_counts()

rn
1     2024234
2     1876171
3     1721621
4     1566205
5     1413568
6     1267150
7     1126682
8      994369
9      873821
10     763014
11     663387
12     573519
13     495054
14     425391
15     365126
16     308955
17     258978
18     212703
19     172044
20     135811
21     105942
22      80790
23      61502
24      45856
25      34113
26      25252
27      18630
28      13752
29      10165
30       7536
31       5454
32       3868
33       2870
34       2043
35       1364
36        966
37        651
38        440
39        295
40        186
41        121
42         87
43         44
44         27
45         19
46         11
47         10
49          7
48          7
50          6
51          5
54          3
52          2
53          2
57          1
56          1
55          1
58          1
Name: count, dtype: int64

## Conclusion  : 
Après une analyse détaillée des colonnes du dataset, nous constatons que la plupart d'entre elles, bien que numériques ou ordinales présentent une structure similaire à celle de variables catégorielles à haute cardinalité, avec une distribution hautement déséquilibrée.

Dans la suite de notre pipeline, nous procéderons à un entraînement sans suppression de données pour préserver l'intégralité des signaux prédictifs, en nous appuyant sur une régularisation renforcée ; toutefois, si les performances sur la validation ou la soumission en ligne s'avèrent insuffisantes,nous reviendrons sur cette approche en supprimant sélectivement des lignes  ou des colonnes à faible valeur ajoutée .

## 2.2 Is columns Id unique ? 

In [17]:
train.shape

(17659833, 62)

In [20]:
len(train["id"].unique())

2892602

## Conclusion 
Après une première inspection du dataset, nous constatons que la table d'entraînement est de taille considérable, avec 17,6 millions de lignes et 62 colonnes, reflétant un historique exhaustif de transactions de crédit. Cependant, une analyse plus approfondie révèle que de nombreux clients (identifiés par l'id) peuvent contracter plusieurs prêts au cours du temps, générant ainsi des redondances structurelles dans les données qui justifient une approche d'agrégation par client pour optimiser l'apprentissage sans perte d'information.

## 2.3 Feature Engineering 


In [2]:
def create_global_aggregations_train_test(train, test):
    """
    Crée les agrégations GLOBALES :
    - Fit sur TRAIN uniquement
    - Transform sur TEST (même stats que train)
    """
    # 1. Calcul des agrégations sur TRAIN
    agg_dict = {
        'rn': 'max',
        'pre_loans_credit_limit': ['mean', 'min', 'max', 'std', 'sum'],
        'pre_loans_outstanding': ['sum', 'mean', 'max'],
        'pre_loans_max_overdue_sum': ['max', 'mean', 'sum'],
        'pre_loans_total_overdue': ['sum', 'max', 'mean'],
        'pre_loans_credit_cost_rate': ['mean', 'std', 'min', 'max'],
        'pre_loans_next_pay_summ': ['sum', 'mean', 'max'],
    }
    
    train_agg = train.groupby('id').agg(agg_dict).reset_index()
    train_agg.columns = [
        'id', 'total_loans',
        'avg_credit_limit', 'min_credit_limit', 'max_credit_limit', 'std_credit_limit', 'sum_credit_limit',
        'total_outstanding', 'avg_outstanding', 'max_outstanding',
        'max_overdue_ever', 'avg_max_overdue', 'total_max_overdue',
        'total_overdue_current', 'max_overdue_current', 'avg_overdue_current',
        'avg_credit_cost_rate', 'std_credit_cost_rate', 'min_credit_cost_rate', 'max_credit_cost_rate',
        'total_next_pay_summ', 'avg_next_pay_summ', 'max_next_pay_summ'
    ]
    
    # Remplacer std NaN par 0
    std_cols = [c for c in train_agg.columns if c.startswith('std_')]
    train_agg[std_cols] = train_agg[std_cols].fillna(0)
    
    # 2. Appliquer les mêmes stats au TEST
    test_agg = test[['id']].merge(train_agg, on='id', how='left')
    
    # Pour les clients ABSENTS du train → imputer avec les moyennes du train
    global_means = train_agg.drop(columns='id').mean()
    test_agg = test_agg.fillna(global_means)
    
    return train_agg, test_agg

In [ ]:
# === 2. Créer les features SANS leakage ===
train_agg, test_agg = create_global_aggregations_train_test(train, test)

# === 3. Enrichir ===
train_enriched = train.merge(train_agg, on='id', how='left')
test_enriched = test.merge(test_agg, on='id', how='left')

# === 4. Nettoyage mémoire ===
del train, test, train_agg, test_agg
gc.collect()

#  3. Model Training

In [ ]:

drop_cols = ['id', 'rn', 'flag']
X = train_enriched.drop(columns=drop_cols)
y = train_enriched['flag']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
    loss_function='Logloss',
    eval_metric='AUC',
    task_type="GPU",
    devices='0',
    
    depth=6,
    iterations=200000,
    learning_rate=0.05,
    l2_leaf_reg=10,
    border_count=128,
    bootstrap_type='Bernoulli',   
    subsample=0.8,
    
    auto_class_weights='Balanced',
    early_stopping_rounds=200,
    use_best_model=True,
    random_seed=42,
    verbose=100,
)

In [4]:
#Attendre 9h sous GPU
print("Entraînement sur GPU...")
model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=100)

Entraînement sur GPU...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6081305	best: 0.6081305 (0)	total: 198ms	remaining: 11h 1m 30s
100:	test: 0.6574856	best: 0.6574856 (100)	total: 16.9s	remaining: 9h 18m 22s
200:	test: 0.6680968	best: 0.6680968 (200)	total: 33.2s	remaining: 9h 10m 26s
300:	test: 0.6746226	best: 0.6746226 (300)	total: 49.8s	remaining: 9h 10m 14s
400:	test: 0.6793954	best: 0.6793954 (400)	total: 1m 6s	remaining: 9h 9m 2s
500:	test: 0.6829485	best: 0.6829485 (500)	total: 1m 22s	remaining: 9h 7m 24s
600:	test: 0.6862406	best: 0.6862406 (600)	total: 1m 38s	remaining: 9h 5m 54s
700:	test: 0.6893761	best: 0.6893761 (700)	total: 1m 54s	remaining: 9h 4m 35s
800:	test: 0.6922550	best: 0.6922550 (800)	total: 2m 11s	remaining: 9h 3m 20s
900:	test: 0.6948094	best: 0.6948094 (900)	total: 2m 27s	remaining: 9h 2m 24s
1000:	test: 0.6974188	best: 0.6974188 (1000)	total: 2m 43s	remaining: 9h 2m 14s
1100:	test: 0.6994663	best: 0.6994663 (1100)	total: 2m 59s	remaining: 9h 1m 9s
1200:	test: 0.7013940	best: 0.7013940 (1200)	total: 3m 16s	remainin

# 4 Evaluation

In [5]:
# === 7. Évaluation ===
train_preds = model.predict_proba(X_train)[:, 1]
val_preds = model.predict_proba(X_val)[:, 1]
print(f"Train AUC: {roc_auc_score(y_train, train_preds):.6f}")
print(f"Val AUC: {roc_auc_score(y_val, val_preds):.6f}")



Train AUC: 0.996961
Val AUC: 0.967863


In [27]:
test_enriched.head()

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag,id_x_rn,total_loans,avg_credit_limit,min_credit_limit,max_credit_limit,std_credit_limit,sum_credit_limit,total_outstanding,avg_outstanding,max_outstanding,max_overdue_ever,avg_max_overdue,total_max_overdue,total_overdue_current,max_overdue_current,avg_overdue_current,avg_credit_cost_rate,std_credit_cost_rate,min_credit_cost_rate,max_credit_cost_rate,total_next_pay_summ,avg_next_pay_summ,max_next_pay_summ
0,1472943,2,10,4,16,16,13,10,10,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0,1472943_x_2,9.0,9.833333,0.0,16.0,5.564770,59.0,20.0,3.333333,5.0,2.0,2.0,12.0,0.0,0.0,0.0,3.500000,3.674235,2.0,11.0,10.0,1.666667,2.0
1,660465,5,6,14,9,7,3,11,0,2,3,0,2,4,6,16,5,4,8,1,0,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,1,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0,660465_x_5,4.0,4.000000,4.0,4.0,0.000000,4.0,3.0,3.000000,3.0,2.0,2.0,2.0,0.0,0.0,0.0,4.000000,0.000000,4.0,4.0,2.0,2.000000,2.0
2,1788193,3,1,0,9,9,7,2,16,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0,1788193_x_3,4.0,8.333333,3.0,12.0,4.725816,25.0,9.0,3.000000,3.0,2.0,2.0,6.0,0.0,0.0,0.0,2.000000,0.000000,2.0,2.0,6.0,2.000000,2.0
3,2767146,3,15,8,1,16,6,13,14,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,9,5,4,0,0,0,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,3,1,0,0,2767146_x_3,20.0,9.750000,2.0,18.0,5.207076,117.0,36.0,3.000000,5.0,2.0,2.0,24.0,0.0,0.0,0.0,6.166667,5.457827,0.0,13.0,29.0,2.416667,6.0
4,2601698,9,19,10,4,8,1,11,4,2,3,0,2,2,6,16,5,4,8,1,1,1,1,1,9,5,4,0,0,0,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,3,1,1,1,2601698_x_9,8.0,6.142857,1.0,16.0,5.014265,43.0,21.0,3.000000,3.0,2.0,2.0,14.0,0.0,0.0,0.0,3.571429,3.505098,1.0,11.0,18.0,2.571429,6.0


# 5 Submission

In [6]:
# === 8. Prédiction sur test ===
X_test = test_enriched[X_train.columns]
test_preds_full = model.predict_proba(X_test)[:, 1]

# === 9. Ajouter les prédictions ===
test_enriched['pred'] = test_preds_full

# === 10. Agréger par id_x_rn (même ordre que le test original) ===
submission_df = test_enriched.groupby('id_x_rn')['pred'].mean().reset_index()

# === 11. Renommer pour la soumission ===
submission_df.columns = ['id', 'target'] 

# === 12. Vérification ===
print(f"Shape submission : {submission_df.shape}")  # (654068, 2)
assert 'id' in submission_df.columns and 'target' in submission_df.columns
assert len(submission_df) == 654068

# === 13. Sauvegarde ===
submission_df.to_parquet("submission2.parquet", index=False)
print("SUBMISSION PRÊTE À ÊTRE SOUMISE !")

Shape submission : (654068, 2)
SUBMISSION PRÊTE À ÊTRE SOUMISE !


# 6 Backup

In [ ]:
import joblib 
joblib.dump(model,'fianl_cat_model.pkl')

# TEST ON PRIVATE DATA
VOUS  POUVEZ EXECUTER CETTE SECTION INDEPENDEMMENT DES AUTRES EN IMPORTANT LE MODELE DEJA ENTRAINE .CEPENDANT VOUS DEVEZ PRENDRE LE SOIN DE BIEN PLACER LES CHEMINS D'ACCES DU MODEL  ET DES DATASSETS .

UNE FOIS CELA FAIT, VEILLEZ EXECUTER TOUTES LES SECTIONS CI-DESSOUS 


In [20]:
pip install df-squeezer

  Preparing metadata (setup.py) ... done
  Created wheel for df-squeezer: filename=df_squeezer-0.0.12-py3-none-any.whl size=3465 sha256=8934224be739a221f3ccdf1e33e32fa19da9197ba9cb20751067594b3ae168ec
  Stored in directory: /root/.cache/pip/wheels/63/ef/7d/95c3a3c2f0210d1385c2a539837efe342ab657dbff8ed76c3f
Successfully built df-squeezer
Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import joblib
import gc
from df_squeezer import df_squeezer
from sklearn.metrics import roc_auc_score

In [15]:
model = joblib.load("/kaggle/input/model-path/fianl_cat_model.pkl") # veillez placer le bon chemin d'acces au model

In [16]:
# Assurez vous de bien  placer les chemin d'acces du datasset d'origine et et du datasset priver 
train = pd.read_parquet("/kaggle/input/datassets/train.parquet") # le chemin  à changer 
private_test  = pd.read_parquet("/kaggle/input/datassets/test.parquet") # le chemin  à changer du test en lechim du private data 


In [22]:
train=df_squeezer(train, edit=True, report=False)
private_test=df_squeezer(private_test, edit=True, report=False)

## Fonction de transformation des datas du train et du test_privé 

In [23]:
def create_global_aggregations_train_test(train, test):
    """
    Crée les agrégations GLOBALES :
    - Fit sur TRAIN uniquement
    - Transform sur TEST (même stats que train)
    """
    # 1. Calcul des agrégations sur TRAIN
    agg_dict = {
        'rn': 'max',
        'pre_loans_credit_limit': ['mean', 'min', 'max', 'std', 'sum'],
        'pre_loans_outstanding': ['sum', 'mean', 'max'],
        'pre_loans_max_overdue_sum': ['max', 'mean', 'sum'],
        'pre_loans_total_overdue': ['sum', 'max', 'mean'],
        'pre_loans_credit_cost_rate': ['mean', 'std', 'min', 'max'],
        'pre_loans_next_pay_summ': ['sum', 'mean', 'max'],
    }
    
    train_agg = train.groupby('id').agg(agg_dict).reset_index()
    train_agg.columns = [
        'id', 'total_loans',
        'avg_credit_limit', 'min_credit_limit', 'max_credit_limit', 'std_credit_limit', 'sum_credit_limit',
        'total_outstanding', 'avg_outstanding', 'max_outstanding',
        'max_overdue_ever', 'avg_max_overdue', 'total_max_overdue',
        'total_overdue_current', 'max_overdue_current', 'avg_overdue_current',
        'avg_credit_cost_rate', 'std_credit_cost_rate', 'min_credit_cost_rate', 'max_credit_cost_rate',
        'total_next_pay_summ', 'avg_next_pay_summ', 'max_next_pay_summ'
    ]
    
    # Remplacer std NaN par 0
    std_cols = [c for c in train_agg.columns if c.startswith('std_')]
    train_agg[std_cols] = train_agg[std_cols].fillna(0)
    
    # 2. Appliquer les mêmes stats au TEST
    test_agg = test[['id']].merge(train_agg, on='id', how='left')
    
    # Pour les clients ABSENTS du train → imputer avec les moyennes du train
    global_means = train_agg.drop(columns='id').mean()
    test_agg = test_agg.fillna(global_means)
    
    return train_agg, test_agg

## Transforamtion du train et private data pour la prediction 

In [24]:
train_agg, private_test_agg = create_global_aggregations_train_test(train, private_test)
train_enriched = train.merge(train_agg, on='id', how='left')
private_test_agg_enriched = private_test.merge(private_test_agg, on='id', how='left')
del train, private_test, train_agg, private_test_agg
gc.collect()

24

## PREDICTION

In [28]:
X_train=train_enriched.drop('flag',axis=1)
y_train=train_enriched['flag']

#### Cette partie de code suppose que votre private data a les meme colones que le train ,la colone 'flag' y compris
#### Si tel est le cas ,enlever les commentaires du code ci-dessous.


In [ ]:
X_private_test=private_test_agg_enriched.drop('flag',axis=1)
y_private_test=private_test_agg_enriched['flag']

## Prédiction finale sur les donnés du train

In [32]:
train_preds = model.predict_proba(X_train)[:, 1]
print(f"Train AUC: {roc_auc_score(y_train, train_preds):.6f}")


Train AUC: 0.991343


## Prédiction finale sur les donnés privées

In [ ]:
private_data_preds = model.predict_proba(X_private_test)[:, 1]
print(f"Val AUC: {roc_auc_score(y_private_test, private_data_preds):.6f}")